To do
------------Algorithmus fertig stellen------------
#Funktion die automatisch den threshold mit dem größten IG berechnet und ausgibt
#Die Funktion muss das über beide Achsen machen
#output: 2 Information gain - Werte und 2 treshold Werte

#dann: Daten in 2 Teilmengen aufteilen, gemäß dem max. IG

#predict Funktion programmieren

#Als Klasse programmieren


#Der Prozess wird wiederholt wie oft ? max depth definieren
---------------Streamlit-------------------------
-slider für den Information Gain

### ID3 Algorithmus programmieren
Anhand des Iris Datensatzes
Laden der ENV : conda create --name myenv --file spec-file.txt

In [93]:
from sklearn.datasets import load_iris
import altair as alt
import pandas as pd
import numpy as np

df = load_iris()
target = df.target
df = df.data[:,2:] #only petal width and height

df = pd.DataFrame(df,columns=['petal length','petal width'])
df['target'] = target
X = df

alt.Chart(X).mark_circle().encode(x='petal length',y='petal width',color='target:N')



alt.Chart(...)

Fig. Iris Datensatz

ID3 Algorithmus

In [94]:
y = X.iloc[:,2]

def calc_Entropy(y):
    H_ = []
    classes, counts = np.unique(y,return_counts=True)

    for i in range(len(classes)):
        p = counts[i]/sum(counts) #Berechnung der Wahrscheinlichkeitfür eine Klasse
        entropy = -p*np.log2(p) #Entrope Formel
        H_.append(entropy)
    return sum(H_)


In [95]:
def calc_information_gain(X_column,y,threshold):
    h_parent = calc_Entropy(y)
    #maske
    mask_r = X_column >= threshold
    mask_l = X_column < threshold
    n_ges = len(y)
    n_r = sum(mask_r)
    n_l = sum(mask_l)
    h_r = calc_Entropy(y[mask_r])
    h_l = calc_Entropy(y[mask_l])
    IG = h_parent -( ((n_r/n_ges)*h_r) + ((n_l/n_ges)*h_l)  )
    return IG
    

In [108]:
def calc_max_IG_column(X_column,y):
    '''
    X_column: DataFrame
        einer Merkmals-Spalte
    y:  npArray
        Mit den Zielwerten für die Entropie Berechnung
        
    return: höchster IG, und den threshold'''
    sorted_values = np.sort(X.iloc[:,0].unique())
    thresholds = (sorted_values[1:] + sorted_values[:-1]) / 2 
    #print(thresholds)
    
    best_IG = -1
    best_IG_threshold = 0
    
    for i in thresholds:
        current_IG = calc_information_gain(X_column,y,i)  
        if current_IG > best_IG:
            best_IG = current_IG
            best_IG_threshold = i

    return best_IG, best_IG_threshold


def calc_max_IG(X,y):
    '''
    X: DataFrame shape=(n_samples,n_features
    y: np-Array shape=(n_samples,n_features)
    return: threshold, axis
    '''
    X_columns_1 = X.iloc[:,0]
    X_columns_2 = X.iloc[:,1]
    max_IG_1, threshold_1 = calc_max_IG_column(X_columns_1,y)
    max_IG_2, threshold_2 = calc_max_IG_column(X_columns_2,y)
    

    
    if max_IG_1 > max_IG_2:
        return threshold_1, 0
    else:
        return threshold_2, 1


def mask(X,threshold,axis):
    '''
    Nomenklatur: X_d_n mit d=Tiefe n=n_Menge'''
    bool_array = X.iloc[:,axis] <= threshold
    return bool_array
    
    #X_1_0 = X[bool_array]
    #X_1_1 = X[-bool_array]
    #return X_1_0, X_1_1



thresh, axis = calc_max_IG(X,y)
mask(X,thresh,axis)

pandas.Series

In [97]:
#Abbruchbedingunen
#max depth ist erreicht
#Sortenreinheit
#IG kann nicht mehr gesteigert werden

In [175]:
class DecisionTree:
    def __init__(self,depth=0,max_depth=3,min_samples=2):
        #self.X = X
        #self.y = y
        self.depth = depth
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.min_samples = min_samples
    class Node:
        def __init__(self,feature=None,threshold=None,left_child=None,right_child=None,depth=None,axis=None):
            self.feature = feature
            self.threshold = threshold
            self.left_child= left_child
            self.right_child = right_child
            self.depth = depth
            self.axis = axis

    def fit(self,X,y):
        self.root = self.build_tree(X, y, depth=0)
    
    def build_tree(self,X,y,depth=0):   
        #print('depth=',depth)
        if depth >= self.max_depth or len(X) <= self.min_samples or int(len(y.unique())) == 1:
            value = int(y.value_counts().idxmax())
            #print('Leaf created: Value =',value)
            return Node(feature=value) #Blatt wird generiert

        thresh, axis_ = calc_max_IG(X,y)
        #print(thresh,axis)
        
        mask_ = mask(X,thresh,axis_)

        
        left_child_ = self.build_tree(X[mask_],y[mask_],depth+1)
        right_child_ = self.build_tree(X[~mask_],y[~mask_],depth+1)

        #print('Node created: threshold =',thresh,'axis=',axis,left_child_,right_child_)
        current_Node = Node(threshold=thresh,left_child=left_child_,right_child=right_child_,axis=axis_)
        return current_Node
        
    def predict(self,X):
        X_array = np.array(X)

        pred_list = []
        for x in X_array:
            pred = self._check_object(x,self.root)
            pred_list.append(pred)
        
        return pred_list
        
        
    def _check_object(self,x,node):
        #Leaf-Check
        if node.left_child == None and node.right_child == None:
            return node.feature
        
        #gesuchter x wert, je nach axis
        _x = x[node.axis]

        if _x <= node.threshold:
            return self._check_object(x,node.left_child)
            
        else:
            return self._check_object(x,node.right_child)
    


In [183]:
Modell = DecisionTree()
Modell.fit(X,y)

predicted = Modell.predict(X)
X['predicted'] = predicted
meine_klassen = [0,1,2]
meine_farben = ['blue','red','green']
alt.Chart(X).mark_circle().encode(x='petal length',y='petal width',color=alt.Color('predicted:N', scale=alt.Scale(domain=meine_klassen, range=meine_farben)))



alt.Chart(...)

In [141]:
def print_tree(node, spacing=""):
    """Gibt den Baum gut lesbar in der Konsole aus."""
    
    # 1. Ist es ein Blatt? (Erkennt man daran, dass es keine Kinder gibt)
    if node.left_child is None and node.right_child is None:
        print(spacing + f"🍁 Blatt: Vorhersage = {node.feature}")
        return

    # 2. Ist es ein Entscheidungsknoten?
    print(spacing + f"🌲 Knoten: Ist Feature [{node.axis}] <= {node.threshold} ?")

    # Linkes Kind (Bedingung ist erfüllt)
    print(spacing + '  --> True:')
    print_tree(node.left_child, spacing + "    ")

    # Rechtes Kind (Bedingung ist nicht erfüllt)
    print(spacing + '  --> False:')
    print_tree(node.right_child, spacing + "    ")

print_tree(Wurzel)

🌲 Knoten: Ist Feature [0] <= 2.45 ?
  --> True:
    🍁 Blatt: Vorhersage = 0
  --> False:
    🌲 Knoten: Ist Feature [1] <= 1.7999999999999998 ?
      --> True:
        🌲 Knoten: Ist Feature [0] <= 4.95 ?
          --> True:
            🍁 Blatt: Vorhersage = 1
          --> False:
            🍁 Blatt: Vorhersage = 2
      --> False:
        🌲 Knoten: Ist Feature [0] <= 4.85 ?
          --> True:
            🍁 Blatt: Vorhersage = 2
          --> False:
            🍁 Blatt: Vorhersage = 2
